# 01 — Create Heatmap

`POST /v1/heatmap` generates a thermal map (GeoJSON tile layer + statistics) over a polygon AOI.

**Plan:** available on both Basic (≤10 mi²) and Premium (≤50 mi²).

Inputs:
- `polygon_aoi` — GeoJSON FeatureCollection (coordinates are `[lon, lat]`)
- `date_time` — `start_date`, `filter_type` (1=single hour, 2=range of hours, 3=single day, 4=range of days), and matching `start_time`/`end_time`/`end_date`
- `granularity` — 60, 80, or 100 meters
- `analytic_type` — `tcm` (snapshot, default), `time_of_measure`, `exceedance`, or `persistence` (see the analysis-heatmaps section below)

### Reference: filter types

The `date_time` payload changes shape depending on `filter_type`. The client builds it for you from the keyword args — pick the variant that matches the question you're asking:

| `filter_type` | Meaning | Required keyword args | Response shape |
|---|---|---|---|
| `1` | **Single hour** | `start_date`, `start_time` | Each tile carries one `temperature` for that hour |
| `2` | **Range of hours** (same day) | `start_date`, `start_time`, `end_time` | Each tile carries one aggregated `temperature` over the range |
| `3` | **Single day** (daily aggregates) | `start_date` | Covers the full day 00:00–23:59 (any `start_time` is ignored). Each tile carries `min_temperature`, `max_temperature`, `average_temperature` (all **°C**). The Enterprise API does **not** return per-hour `'00'..'23'` fields. |
| `4` | **Range of days** (window ≤ ~31 days) | `start_date`, `end_date` | Each tile carries aggregates over the multi-day window |

The use-case notebooks under `notebooks/use_cases/` all call with `filter_type=3` — that single call gives them both the daily peak (for ranking) and the full diurnal series (for peak-hour and swing analysis). Pick `filter_type=1` only when you genuinely want one snapshot.

```python
# filter_type=2 — average over the afternoon peak window
client.create_heatmap(polygon_aoi=AOI, start_date='2024-07-15',
                     start_time='12:00', end_time='17:00',
                     filter_type=2, granularity=100)

# filter_type=3 — full single-day capture (recommended for use cases)
client.create_heatmap(polygon_aoi=AOI, start_date='2024-07-15',
                     start_time='14:00',
                     filter_type=3, granularity=100)
```

### Reference: `granularity`

Spatial resolution of the output tiles, in meters. Trade-off: smaller value → more tiles → richer detail → longer runtime and higher credit cost.

| `granularity` | Approx. tile count for the bundled ~104 km² San Jose AOI | When to use |
|---|---|---|
| `100` | ~10,000 | Fast iteration / large AOIs |
| `80` | ~16,500 | Default for the use-case notebooks — good balance |
| `60` | ~28,000 | Fine-grained block-level analysis |

### Reference: response schema

`client.create_heatmap(...)` returns `{"activity_id": str, "result": dict}`. The `result` carries:

- **`stats_data`** — AOI-wide aggregates (e.g. `Temperature_stats`, `Overall_temperature_distribution`).
- **`map_data`** — a GeoJSON `FeatureCollection`. Each feature has:
  - `geometry` — a `Polygon` outlining the tile.
  - `properties.tile_id` — stable identifier for the tile.
  - `properties.temperature` — for `filter_type=1` / `2` (single value).
  - `properties.average_temperature` / `min_temperature` / `max_temperature` — daily °C aggregates for `filter_type=3` / `4` (the Enterprise API does not emit per-hour `'00'..'23'` fields).
  - All tile temperatures are **°C** — no conversion needed.

> **Units:** the Enterprise API delivers tile temperatures in **°C** — use them directly, no conversion needed. (The *Dashboard* product converts to °F for display, but this API does not.)


In [13]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
from fortyguard.samples import MANHATTAN_POLYGON

client = FortyGuardClient()

In [ ]:
response = client.create_heatmap(
    polygon_aoi=MANHATTAN_POLYGON,
    start_date='2024-07-15',
    start_time='14:00',
    filter_type=1,     # single hour
    granularity=60,
)

activity_id = response['activity_id']
result = response['result']
print(f'activity_id: {activity_id}')
print(f'result keys: {list(result.keys())}')

Submitted -> activity_id=665f5b46-f705-4506-8a7f-681381105337
  status: processing
  status: processing
  status: processing


KeyboardInterrupt: 

In [ ]:
# Look at the aggregated statistics.
stats = result.get('stats_data', {})
temp_stats = stats.get('Temperature_stats') or stats.get('temperature_stats') or {}
print('Temperature stats:')
for key, value in temp_stats.items():
    print(f'  {key:>20}: {value}')

In [ ]:
# Plot the temperature distribution if present.
import matplotlib.pyplot as plt

dist = stats.get('Overall_temperature_distribution') or stats.get('overall_temperature_distribution')
if dist:
    plt.figure(figsize=(8, 3))
    plt.hist(dist, bins=40, color='tomato', edgecolor='white')
    plt.xlabel('Temperature (°C)'); plt.ylabel('Tile count')
    plt.title('Heatmap tile temperature distribution'); plt.tight_layout(); plt.show()
else:
    print('No distribution data returned — inspect `stats` above for alternate fields.')

In [ ]:
# Visualise the GeoJSON tiles on a Folium map, coloured by temperature.
import folium

map_data = result.get('map_data')
if map_data and map_data.get('features'):
    temps = [f['properties'].get('temperature') for f in map_data['features'] if 'temperature' in f.get('properties', {})]
    lo, hi = (min(temps), max(temps)) if temps else (0, 1)
    
    def _style(feature):
        t = feature['properties'].get('temperature', lo)
        frac = 0 if hi == lo else (t - lo) / (hi - lo)
        r = int(255 * frac); b = int(255 * (1 - frac))
        return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.65, 'weight': 0}
    
    centroid = MANHATTAN_POLYGON['features'][0]['geometry']['coordinates'][0][0]
    fmap = folium.Map(location=[centroid[1], centroid[0]], zoom_start=14, tiles='cartodbpositron')
    folium.GeoJson(map_data, style_function=_style).add_to(fmap)
    fmap
else:
    print('No map_data features present — result shape may differ for this request.')

### Analysis heatmaps (`analytic_type`)

The snapshot (`tcm`) answers *"how hot is each tile?"*. Over a multi-hour (`filter_type=2`) or multi-day (`filter_type=4`) window, three **analysis heatmaps** answer richer questions from the same underlying time series — selected with the `analytic_type` flag:

| `analytic_type` | Each tile shows | Units | Extra params |
|---|---|---|---|
| `tcm` *(default)* | Snapshot temperature | °C | — |
| `time_of_measure` | **UTC hour-of-day** (0–23) of the tile's peak | hour | — |
| `exceedance` | **Count of hours** the tile spends past `threshold` | hour | `threshold` (°C), `direction` |
| `persistence` | **Longest continuous run** of such hours | hour | `threshold` (°C), `direction` |

`direction` is `'above'` or `'below'`. Both `threshold` and `direction` are required for `exceedance` and `persistence`, and ignored for `tcm` / `time_of_measure`.

> **`threshold` is in °C** (default 30 °C on the API side) — consistent with the `tcm` tile temperatures, which are also returned in **°C**.

> **`exceedance` counts hours, not degree-hours.** A value of `6.0` means the tile spent six hours past the threshold.

#### Response shape — different from `tcm`

The three analysis types return **one `value` per tile** instead of the `tcm` temperature fields:

- `map_data.features[].properties` &rarr; `{ tile_id, value }`
- `stats_data` &rarr; `{ activity_id, analytic_type, units, n_cells, min, max, mean }`

So the `properties.temperature` / `'00'..'23'` / `min_temperature` fields documented in the *response schema* reference above apply to **`tcm` only**. On an analysis heatmap, read `properties.value` and interpret it with `stats_data.units`.

In [ ]:
# Exceedance heatmap: how many hours each tile spends above 35 C over a week.
exceedance = client.create_heatmap(
    polygon_aoi=MANHATTAN_POLYGON,
    start_date='2024-07-15',
    end_date='2024-07-21',
    filter_type=4,               # range of days
    analytic_type='exceedance',
    threshold=35.0,              # degrees CELSIUS (not F)
    direction='above',
    granularity=100,
)

ex_result = exceedance['result']
ex_stats = ex_result['stats_data']
print(f"activity_id : {exceedance['activity_id']}")
print(f"analytic_type: {ex_stats['analytic_type']}  |  units: {ex_stats['units']}")
print(f"cells        : {ex_stats['n_cells']}")
print(f"hours past 35 C -> min {ex_stats['min']}  mean {ex_stats['mean']:.2f}  max {ex_stats['max']}")

# Each analysis tile carries `value` (NOT `temperature`).
first = ex_result['map_data']['features'][0]['properties']
print(f"first tile   : {first}")